In [ ]:
from pathlib import Path
from datetime import date, datetime, timedelta, timezone

import polars as pl

In [ ]:
SCANNER_CONFIG = {
    "data_folder_path": Path("D:/TradingData/massive_flatfiles/us_stock_sip/minutes_agg_v1/2024/05"),
    "scan_date": "2024-05-01",
    "box_start_time": "09:30",
    "box_end_time": "09:35",
    # New York is UTC-4 during May 2024. Use -5 for EST dates outside daylight saving time.
    "market_utc_offset_hours": -4,
    "lookback_daily_files": 20,
    "daily_lookback_root": None,
    "min_box_bars": 1,
    "show_only_passing": False,

    "min_price": 0.5,
    "max_price": 100.0,
    "min_avg_daily_volume": 0,
    "min_gap_fraction": 0.0,
    "min_box_relative_volume": 0.0,
    "min_box_range_atr_fraction": 0.0,
    "max_box_range_atr_fraction": None,
    "min_close_location": 0.0,
    "min_green_body_fraction": 0.0,

    "expected_box_daily_volume_share": 0.02,
    "ideal_range_atr_fraction": 0.30,
    "liquidity_score_volume": 10_000_000,

    "output_columns": [
        "ticker", "final_score", "passes_qc_setup_filter", "box_high", "box_mid", "box_low",
        "box_open", "box_close", "box_volume", "box_transactions", "box_bar_count",
        "box_daily_volume_share", "box_relative_volume", "box_range", "box_range_atr_fraction", "gap_fraction",
        "avg_daily_volume_14", "atr14", "previous_close", "close_location", "green_body_fraction",
        "relative_volume_score", "range_quality_score", "gap_score", "liquidity_score",
        "box_first_bar_time", "box_last_bar_time",
    ],
}

## Opening Box Scanner

`run_opening_box_scanner(config)` is the main frontend function. Pass all scanner settings in one dict; the downstream functions receive the normalized config and do not read notebook globals.

The market session conversion avoids `ZoneInfo`; configure `market_utc_offset_hours` explicitly for the date being scanned. For May 2024 New York market data, use `-4`.

In [ ]:
def normalize_scanner_config(config: dict) -> dict:
    """Return a scanner config with paths normalized and optional defaults filled."""
    cfg = dict(config)
    cfg["data_folder_path"] = Path(cfg["data_folder_path"])
    cfg["scan_date"] = str(cfg["scan_date"])
    cfg["daily_lookback_root"] = (
        cfg["data_folder_path"].parent if cfg.get("daily_lookback_root") is None else Path(cfg["daily_lookback_root"])
    )
    cfg["lookback_daily_files"] = int(cfg.get("lookback_daily_files", 20))
    cfg["min_box_bars"] = int(cfg.get("min_box_bars", 1))
    cfg["market_utc_offset_hours"] = float(cfg["market_utc_offset_hours"])
    cfg["show_only_passing"] = bool(cfg.get("show_only_passing", False))
    return cfg


def parse_hhmm(hhmm: str) -> tuple[int, int]:
    hour, minute = hhmm.split(":", maxsplit=1)
    return int(hour), int(minute)


def date_from_path(path: Path) -> date:
    # Handles names like 2024-05-01.csv.gz.
    return date.fromisoformat(path.stem.removesuffix(".csv"))


def market_time_to_utc_ns(cfg: dict, hhmm: str) -> int:
    hour, minute = parse_hhmm(hhmm)
    local_dt = datetime.fromisoformat(cfg["scan_date"]).replace(hour=hour, minute=minute)
    utc_dt = local_dt - timedelta(hours=cfg["market_utc_offset_hours"])
    return int(utc_dt.replace(tzinfo=timezone.utc).timestamp() * 1_000_000_000)


def add_bar_time_columns(frame: pl.DataFrame, cfg: dict) -> pl.DataFrame:
    offset_ns = int(cfg["market_utc_offset_hours"] * 60 * 60 * 1_000_000_000)
    return (
        frame.with_columns((pl.col("window_start") + offset_ns).alias("market_window_start"))
        .with_columns(
            pl.from_epoch("window_start", time_unit="ns").alias("bar_time_utc"),
            pl.from_epoch("market_window_start", time_unit="ns").alias("bar_time_market"),
        )
        .drop("market_window_start")
    )


def build_opening_box(minute_bars: pl.DataFrame, cfg: dict) -> pl.DataFrame:
    start_ns = market_time_to_utc_ns(cfg, cfg["box_start_time"])
    end_ns = market_time_to_utc_ns(cfg, cfg["box_end_time"])

    box_bars = (
        minute_bars.filter(
            (pl.col("window_start") >= start_ns)
            & (pl.col("window_start") < end_ns)
        )
        .sort(["ticker", "window_start"])
        .pipe(add_bar_time_columns, cfg)
    )

    return (
        box_bars.group_by("ticker")
        .agg(
            pl.col("open").first().alias("box_open"),
            pl.col("close").last().alias("box_close"),
            pl.col("high").max().alias("box_high"),
            pl.col("low").min().alias("box_low"),
            pl.col("volume").sum().alias("box_volume"),
            pl.col("transactions").sum().alias("box_transactions"),
            pl.len().alias("box_bar_count"),
            pl.col("bar_time_market").first().alias("box_first_bar_time"),
            pl.col("bar_time_market").last().alias("box_last_bar_time"),
        )
        .filter(pl.col("box_bar_count") >= cfg["min_box_bars"])
        .with_columns(
            (pl.col("box_high") - pl.col("box_low")).alias("box_range"),
            ((pl.col("box_high") + pl.col("box_low")) / 2).alias("box_mid"),
            pl.when(pl.col("box_high") > pl.col("box_low"))
            .then((pl.col("box_close") - pl.col("box_low")) / (pl.col("box_high") - pl.col("box_low")))
            .otherwise(None)
            .alias("close_location"),
            pl.when(pl.col("box_high") > pl.col("box_low"))
            .then((pl.col("box_close") - pl.col("box_open")) / (pl.col("box_high") - pl.col("box_low")))
            .otherwise(None)
            .clip(0, 1)
            .alias("green_body_fraction"),
        )
    )


def prior_data_paths(cfg: dict) -> list[Path]:
    scan_day = date.fromisoformat(cfg["scan_date"])
    paths = []
    for csv_path in sorted(cfg["daily_lookback_root"].glob("*/*.csv.gz"), reverse=True):
        try:
            session_date = date_from_path(csv_path)
        except ValueError:
            continue
        if session_date < scan_day:
            paths.append(csv_path)
        if len(paths) >= cfg["lookback_daily_files"]:
            break
    return list(reversed(paths))


def load_prior_daily_stats(cfg: dict) -> pl.DataFrame:
    scans = []
    for csv_path in prior_data_paths(cfg):
        session_date = date_from_path(csv_path)
        scans.append(
            pl.scan_csv(csv_path)
            .group_by("ticker")
            .agg(
                pl.col("volume").sum().alias("day_volume"),
                pl.col("high").max().alias("day_high"),
                pl.col("low").min().alias("day_low"),
                pl.col("close").last().alias("day_close"),
            )
            .with_columns(pl.lit(session_date).alias("session_date"))
        )

    if not scans:
        return pl.DataFrame(
            schema={
                "ticker": pl.String,
                "avg_daily_volume_14": pl.Float64,
                "atr14": pl.Float64,
                "previous_close": pl.Float64,
            }
        )

    return (
        pl.concat(scans)
        .sort(["ticker", "session_date"])
        .with_columns((pl.col("day_high") - pl.col("day_low")).alias("day_range"))
        .group_by("ticker")
        .agg(
            pl.col("day_volume").tail(14).mean().alias("avg_daily_volume_14"),
            pl.col("day_range").tail(14).mean().alias("atr14"),
            pl.col("day_close").last().alias("previous_close"),
        )
        .collect()
    )


def min_threshold_ok(column: str, threshold) -> pl.Expr:
    if threshold is None or threshold <= 0:
        return pl.lit(True)
    return pl.col(column) >= threshold


def max_threshold_ok(column: str, threshold) -> pl.Expr:
    if threshold is None or threshold >= 99_999:
        return pl.lit(True)
    return pl.col(column) <= threshold


def score_boxes(boxes: pl.DataFrame, daily_stats: pl.DataFrame, cfg: dict) -> pl.DataFrame:
    expected_volume_share = cfg["expected_box_daily_volume_share"]
    ideal_range_atr_fraction = cfg["ideal_range_atr_fraction"]
    liquidity_score_volume = cfg["liquidity_score_volume"]

    scored = boxes.join(daily_stats, on="ticker", how="left").with_columns(
        pl.when(pl.col("previous_close") > 0)
        .then((pl.col("box_open") - pl.col("previous_close")) / pl.col("previous_close"))
        .otherwise(None)
        .alias("gap_fraction"),
        pl.when(pl.col("avg_daily_volume_14") > 0)
        .then(pl.col("box_volume") / pl.col("avg_daily_volume_14"))
        .otherwise(None)
        .alias("box_daily_volume_share"),
        pl.when(pl.col("atr14") > 0)
        .then(pl.col("box_range") / pl.col("atr14"))
        .otherwise(None)
        .alias("box_range_atr_fraction"),
    )

    scored = scored.with_columns(
        pl.when(pl.lit(expected_volume_share) > 0)
        .then(pl.col("box_daily_volume_share") / expected_volume_share)
        .otherwise(None)
        .alias("box_relative_volume")
    )

    scored = scored.with_columns(
        pl.when(pl.lit(expected_volume_share) > 0)
        .then(pl.col("box_relative_volume").clip(0, 3) / 3)
        .otherwise(pl.lit(0.0))
        .fill_null(0)
        .alias("relative_volume_score"),
        pl.when(pl.lit(ideal_range_atr_fraction) > 0)
        .then((1 - ((pl.col("box_range_atr_fraction") - ideal_range_atr_fraction).abs() / ideal_range_atr_fraction)).clip(0, 1))
        .otherwise(pl.lit(0.0))
        .fill_null(0)
        .alias("range_quality_score"),
        pl.col("gap_fraction").clip(0, 0.20).fill_null(0).alias("gap_score"),
        pl.when(pl.lit(liquidity_score_volume) > 0)
        .then((pl.col("avg_daily_volume_14") / liquidity_score_volume).clip(0, 1))
        .otherwise(pl.lit(0.0))
        .fill_null(0)
        .alias("liquidity_score"),
    )

    return scored.with_columns(
        (
            35 * pl.col("relative_volume_score")
            + 20 * pl.col("range_quality_score")
            + 15 * pl.col("close_location").fill_null(0)
            + 10 * pl.col("green_body_fraction").fill_null(0)
            + 10 * pl.col("gap_score")
            + 10 * pl.col("liquidity_score")
        ).alias("final_score"),
        (
            (pl.col("box_close") >= cfg["min_price"])
            & (pl.col("box_close") <= cfg["max_price"])
            & (pl.col("box_high") > pl.col("box_low"))
            & min_threshold_ok("avg_daily_volume_14", cfg["min_avg_daily_volume"])
            & min_threshold_ok("gap_fraction", cfg["min_gap_fraction"])
            & min_threshold_ok("box_daily_volume_share", cfg["min_box_relative_volume"])
            & min_threshold_ok("box_range_atr_fraction", cfg["min_box_range_atr_fraction"])
            & max_threshold_ok("box_range_atr_fraction", cfg["max_box_range_atr_fraction"])
            & min_threshold_ok("close_location", cfg["min_close_location"])
            & min_threshold_ok("green_body_fraction", cfg["min_green_body_fraction"])
        ).alias("passes_qc_setup_filter"),
    ).sort("final_score", descending=True)


def run_opening_box_scanner(config: dict) -> dict[str, pl.DataFrame]:
    cfg = normalize_scanner_config(config)
    scan_path = cfg["data_folder_path"] / f"{cfg['scan_date']}.csv.gz"
    scan_minute_bars = pl.read_csv(scan_path)
    box_df = build_opening_box(scan_minute_bars, cfg)
    daily_stats_df = load_prior_daily_stats(cfg)
    scanner_df = score_boxes(box_df, daily_stats_df, cfg)

    output_df = scanner_df
    if cfg["show_only_passing"]:
        output_df = output_df.filter(pl.col("passes_qc_setup_filter"))

    output_columns = [col for col in cfg["output_columns"] if col in output_df.columns]
    return {
        "config": cfg,
        "box_df": box_df,
        "daily_stats_df": daily_stats_df,
        "scanner_df": scanner_df,
        "scanner_output_df": output_df.select(output_columns),
    }

In [ ]:
scanner_result = run_opening_box_scanner(SCANNER_CONFIG)
scanner_output_df = scanner_result["scanner_output_df"]
scanner_output_df